# Local GPT-2 comparison

Fetch the pinned model explicitly before execution; nothing is downloaded here. Twenty continuations use two prompts, five policies and seeds 7/42. These outputs are unverified samples, not a factuality/quality benchmark.

In [ ]:
import os
from pathlib import Path
import uuid
project = Path.cwd()
if not (project/'configs').is_dir(): project = project.parent
assert (project/'configs').is_dir(), 'Launch from the project or notebooks directory'
os.chdir(project)
run = project/'artifacts'/('notebook-'+uuid.uuid4().hex[:12])

model_dir = Path(os.environ.get('LLM_DECODING_MODEL',str(project/'assets/gpt2')))
assert model_dir.is_dir(), 'Run fetch-model explicitly first'

In [ ]:
from llm_decoding.workflow import compare
summary = compare(model_dir,project/'configs/comparison.json',run)
print(summary)

In [ ]:
from IPython.display import display
from PIL import Image
display(Image.open(run/'first-token.png'))
display(Image.open(run/'continuations.png'))

## Reproducibility and cache check

The implementation uses raw forward calls. Reusing an encoder/past cache does not change the intended greedy policy; numerical behavior is checked for this actual model/example, not every backend.

In [ ]:
import json
from dataclasses import replace
from llm_decoding.engine import load_local,generate
from llm_decoding.sampling import SamplingConfig
model,tokenizer,manifest = load_local(model_dir)
records = [json.loads(line) for line in (run/'continuations.jsonl').read_text().splitlines()]
sample = next(row for row in records if row['prompt_index']==1 and row['strategy_name']=='combined' and row['config']['seed']==42)
config = SamplingConfig(**sample['config'])
repeated = generate(model,tokenizer,sample['prompt'],config)
assert repeated['generated_token_ids'] == sample['generated_token_ids']
short = SamplingConfig(strategy='greedy',max_new_tokens=8)
cached = generate(model,tokenizer,records[0]['prompt'],short)
uncached = generate(model,tokenizer,records[0]['prompt'],replace(short,use_cache=False))
assert cached['generated_token_ids'] == uncached['generated_token_ids']
print('Seeded repeat and checked cache behavior match.')

In [ ]:
for record in records:
    if record['prompt_index']==1 and record['config']['seed']==42:
        print(record['strategy_name'],repr(record['continuation']))

Do not interpret generated numerical/factual claims as verified answers. Greedy records repeat across seeds, later policies see different contexts, and token distinctness is not quality. All measured runs stop at 24-token budgets; EOS behavior is exercised separately in regression fixtures. See the model/use card and full step traces.